# SLC46A1 engraftment MD, part 3: the three arms compared

Taner Karagol and Alper Karagol.

Open this once the part 2 notebooks have run. It reads every
`md/results/<arm>_r<n>_report.json` on Drive, pools the replicates of each arm and
states every difference against the chicken wild type.

## What it does

For each observable the part 2 runs measured, it reports the wild type's mean over
its replicates with their spread, then each humanised arm's mean, its gap from the
wild type, and that gap expressed in the between-replicate spread of the two arms.

## What it does not do

It issues no verdict. No threshold was stated before these runs, the only scale a
difference can be put on is the spread between replicates of one arm, and with a
handful of replicates that spread is itself poorly determined. A gap of a few
replicate standard deviations is a quantity to look at.

The numbers these runs produce say nothing about folding, binding or transport. The
humanised surrogate requires experimental validation of all three before use in
pharmacological screening.

## What it refuses to do

An arm with fewer replicates than `MIN_REPLICATES`, since a difference between arms
could not then be told from one between replicates of the same arm. Replicates of one
arm sharing a seed, which are the same trajectory twice. Replicates of one arm that
resolved different numbering shifts, whose per-position values are not on one
numbering. A set of reports with no wild type in it.


In [ ]:
PROJECT_DIR     = ""
DRIVE_ROOT      = "/content/drive/MyDrive/MIT/All Project Files"
PROJECT_FOLDER  = "SLC46A1 engraftment"
MIN_NS          = 20
MIN_REPLICATES  = 2

from google.colab import drive
drive.mount('/content/drive')

import os, json, glob, datetime, collections
import numpy as np

PROJECT = PROJECT_DIR.strip() or os.path.join(DRIVE_ROOT, PROJECT_FOLDER)
MD = os.path.join(PROJECT, "md")
RESULTS = os.path.join(MD, "results")
INPUTS = os.path.join(MD, "inputs")
SPEC = json.load(open(os.path.join(INPUTS, "build_spec.json")))
ARMS = list(SPEC["builds"])
REFERENCE = "wt"
if REFERENCE not in ARMS:
    raise SystemExit("STOP: the spec lists the arms %s and the comparison is stated "
                     "against %s" % (ARMS, REFERENCE))
WATCH = sorted(SPEC["watch"])
print("project", PROJECT)
print("arms", ", ".join(ARMS), "| reference", REFERENCE, "| watched", WATCH)


In [ ]:
reports = collections.defaultdict(list)
short, seen = [], []
for p in sorted(glob.glob(os.path.join(RESULTS, "*_report.json"))):
    r = json.load(open(p))
    if "arm" not in r or r["arm"] not in ARMS:
        continue
    seen.append((r["arm"], r.get("replicate"), r.get("ns")))
    if r.get("ns", 0) < MIN_NS:
        short.append((os.path.basename(p), r.get("ns")))
        continue
    reports[r["arm"]].append(r)
if not reports:
    raise SystemExit("STOP: no report in %s reaches %g ns. Reports found: %s"
                     % (RESULTS, MIN_NS, seen or "none"))
for arm in ARMS:
    print("  %-10s %d replicate(s) at or above %g ns, lengths %s"
          % (arm, len(reports[arm]), MIN_NS,
             [round(r["ns"], 1) for r in reports[arm]] or "none"))
if short:
    print("  left out as too short: %s" % short)

thin = [a for a in ARMS if len(reports[a]) < MIN_REPLICATES]
if thin:
    raise SystemExit("STOP: %s have fewer than %d replicates at or above %g ns, so "
                     "there is no between-replicate spread to compare against and a "
                     "difference between arms could not be told from one between "
                     "replicates of the same arm." % (thin, MIN_REPLICATES, MIN_NS))
if REFERENCE not in reports:
    raise SystemExit("STOP: no %s replicate reached %g ns, and every comparison here "
                     "is stated against it" % (REFERENCE, MIN_NS))

seeds = {a: sorted(r.get("seed") for r in reports[a]) for a in ARMS}
for a in ARMS:
    if len(set(seeds[a])) != len(seeds[a]):
        raise SystemExit("STOP: %s has replicates sharing a seed, %s. They are the "
                         "same trajectory twice, so their spread is not a measure of "
                         "anything." % (a, seeds[a]))
shifts = {a: {r["numbering_shift"] for r in reports[a]} for a in ARMS}
if any(len(v) != 1 for v in shifts.values()):
    raise SystemExit("STOP: replicates of one arm resolved different numbering "
                     "shifts, %s, so their per-position values are not on one "
                     "numbering." % shifts)
ns_used = min(r["ns"] for a in ARMS for r in reports[a])
print("\ncomparing at the shortest length that every arm reached, %.1f ns" % ns_used)


def series(arm, path):
    out = []
    for r in reports[arm]:
        v = r
        for k in path:
            if v is None:
                break
            v = v.get(k) if isinstance(v, dict) else None
        if v is not None:
            out.append(float(v))
    return out


OBSERVABLES = [("backbone RMSD, A", ["rmsd_A", "mean"]),
               ("radius of gyration, A", ["radius_of_gyration_A", "mean"]),
               ("RMSF over the chain, A", ["rmsf_A_whole_chain", "mean"]),
               ("area per lipid, A2", ["box", "area_per_lipid_A2", "mean"]),
               ("thermostat, K", ["box", "temperature_K", "mean"])]
OBSERVABLES += [("RMSF at chicken %d, A" % w, ["rmsf_A_at_watched", str(w)])
                for w in WATCH]
pair_names = sorted({k for a in ARMS for r in reports[a]
                     for k in r.get("pair_distances_A", {})})
OBSERVABLES += [("%s, A" % nm, ["pair_distances_A", nm, "mean"]) for nm in pair_names]

rows = []
for label, path in OBSERVABLES:
    ref = series(REFERENCE, path)
    if len(ref) < MIN_REPLICATES:
        continue
    row = {"observable": label, "path": path,
           REFERENCE: {"mean": float(np.mean(ref)), "sd": float(np.std(ref, ddof=1)),
                       "n": len(ref)}}
    for arm in ARMS:
        if arm == REFERENCE:
            continue
        v = series(arm, path)
        if len(v) < MIN_REPLICATES:
            continue
        sd = float(np.sqrt((np.var(ref, ddof=1) + np.var(v, ddof=1)) / 2.0))
        gap = float(np.mean(v) - np.mean(ref))
        row[arm] = {"mean": float(np.mean(v)), "sd": float(np.std(v, ddof=1)),
                    "n": len(v), "gap_from_%s" % REFERENCE: gap,
                    "gap_in_replicate_sd": (float(gap / sd) if sd > 1e-9 else None)}
    rows.append(row)

print("\n%-28s %-22s %-22s %s"
      % ("observable", REFERENCE, ARMS[0] if ARMS[0] != REFERENCE else ARMS[1], ""))
others = [a for a in ARMS if a != REFERENCE]
for row in rows:
    ref = row[REFERENCE]
    line = "%-28s %7.2f +/- %-6.2f" % (row["observable"][:28], ref["mean"], ref["sd"])
    for arm in others:
        if arm in row:
            d = row[arm]
            z = d["gap_in_replicate_sd"]
            line += "  | %-10s %7.2f +/- %-5.2f  gap %+6.2f (%s)" % (
                arm, d["mean"], d["sd"], d["gap_from_%s" % REFERENCE],
                "%+.1f sd" % z if z is not None else "sd ~ 0")
    print(line)

out = {"written": datetime.datetime.now().isoformat(timespec="seconds"),
       "reference": REFERENCE, "arms": ARMS, "ns_compared": ns_used,
       "min_ns": MIN_NS, "replicates": {a: len(reports[a]) for a in ARMS},
       "seeds": seeds, "watched": WATCH, "rows": rows,
       "what_this_bounds":
           ("Each arm contributes its replicates, and the only scale a difference is "
            "stated in is the spread between replicates of the same arm. With this "
            "many replicates that spread is itself poorly determined, so a gap of a "
            "few replicate standard deviations is a quantity to look at and not a "
            "result. Nothing here tests folding, binding or transport, and the "
            "humanised surrogate still requires experimental validation of all "
            "three before use in pharmacological screening.")}
json.dump(out, open(os.path.join(RESULTS, "compare_report.json"), "w"), indent=2)
print("\nwrote %s" % os.path.join(RESULTS, "compare_report.json"))
print(out["what_this_bounds"])
